In [1]:
%load_ext autoreload
%autoreload 2

```{toctree}
:maxdepth: 1
:caption: Contents:

# Launching SWAN Cases with `swanpy`

## Imports

In [2]:
import warnings
warnings.filterwarnings('ignore')

from oceanicospy.models.swanpy import Initializer
from oceanicospy.models.swanpy.preprocess import GridMaker, BathyMaker, WindForcing, BottomFrictionProcessor, BoundaryConditions
from oceanicospy.models.swanpy.execution import CaseRunner

## Stationary case:

This notebook walks through a complete SWAN **stationary** modelling workflow using the `swanpy` subpackage inside `oceanicospy`. The example is based on a stationary case for the San Andres island in Colombia.

```{note}
The workflow assumes that input data (bathymetry `.bot`, friction `.fric` and output points `.csv`) are already placed in the appropriate `input/domain_01/` folder inside the case directory.

### Case parameters definition

Three main parameters are required to set up a SWAN case:

| Parameter | Description |
|---|---|
| `path_case` | Root directory that will hold `input/`, `run/`, `output/`, `pros/` |
| `dict_ini_data` | Metadata dictionary consumed by `Initializer` |
| `ini_date` / `end_date` | Simulation window |


The initialization metadata dictionary (`dict_ini_data`) is a key component of the workflow, as it contains all the necessary information to retrieve and process input data (e.g., ERA5 files). This dictionary is expetected to have the following keys:

- ``name``: A string identifier for the case (e.g., "hurricane_matthew_guajira_2016").
- ``case_number`` (optional): An integer to differentiate between multiple cases (e.g., 1, 2, 3).
- ``case_description``: A brief description of the case (e.g., "Stationary", "Non-stationary").
- ``stat_id``: An integer indicating the type of run (0 for non-stationary, 1 for stationary). 
- ``nested_domains``: number of nested domains (e.g., 0, 1, 2) if applicable.
- ``parent_domains``: a dictionary specifying the parent domain for each nested domain (e.g., {"domain_02": "domain_01", "domain_03": "domain_02"}). If there are no nested domains, this can be set to None or an empty dictionary.

 The expected structure of this dictionary is detailed in the `Initializer` documentation.

In [3]:
path_case = '../data/model_runs/stationary_swan/'

dict_ini_data = dict(
    name            = 'Stationary case for San Andres',
    case_number     = 1,
    case_description= 'Example',
    stat_id         = 1,
    number_domains  = 1
)

```{hint}
`stat_id` convention
* `stat_id = 0` → **non-stationary** run  
* `stat_id = 1` → **stationary** run  ← _used in this notebook_

### Initial setup

`Initializer` class creates the project directory tree and writes the baseline SWAN configuration file (`run.swn`) from a template. The tree has a structure like the following, where `domain_01` is the main domain (and only domain in this case):

```
path_case/
├── input/
│   └── domain_01/          ← place ERA5 .nc and bathymetry .dat files here
├── pros/
├── run/
│   └── domain_01/
│       └── run.swn         ← generated from template
└── output/
    └── domain_01/
```



All of this code is expected to be in a python file inside the `pros/` folder so `pros/` is the working directory when running the script and it should be already created before calling `Initializer`. The folder `input/domain_0X/` should also be created and populated with the necessary input files (ERA5 NetCDF files and bathymetry `.dat` file) before running the script.

```{note}

```{note}
Calling `create_folders_l2()` is only required when running **nested** domains.  
For a single-domain run `create_folders_l1()` is sufficient.

In [4]:
case = Initializer(
    root_path    = path_case,
    dict_ini_data= dict_ini_data,
)

case.create_folders()    # creates all project folders (data/, output/, etc.)
case.replace_ini_data()     # writes run.swn from the STAT template

*** Initializing SWAN model ***


	*** Creating project folder structure ***


	*** Copying base SWAN configuration file into run folder ***



### Creating the grid

`GridMaker` is a class inside `swanpy` that populates the `CGRID` and `INPGRID` sections of configuration file `run.swn`. 

One method is available to obtain the grid parameters:

| Method | When to use |
|---|---|
| `params_from_bathy()` | Derive extent and resolution automatically from the `.dat` bathymetry file |

```{note}
If the params_from_bathy() method is used, the grid parameters will be automatically derived from the `.dat` bathymetry file. In this case, the user must ensure that the bathymetry file is correctly formatted and placed in the appropriate `input/domain_0X/` folder before running the script.

When the user provides the grid parameters, the following keys are expected in the `grid_info` dictionary:
- `lon_ll_corner`: longitude of the lower-left corner of the grid
- `lat_ll_corner`: latitude of the lower-left corner of the grid
- `x_extent`: total extent of the grid in the x-direction (longitude)
- `y_extent`: total extent of the grid in the y-direction (latitude)
- `nx` : number of grid cells in the x-direction
- `ny`: number of grid cells in the y-direction


In [5]:
DOMAIN_ID = 1   # domain identifier (1-indexed)

case_grid = GridMaker(
    init         = case,
    domain_number= DOMAIN_ID,
    dict_info    = {'lon_ll_corner': 278.2265167, 
                    'lat_ll_corner': 12.4413166, 
                    'x_extent': 0.13185, 
                    'y_extent': 0.1908, 
                    'nx': 293,
                    'ny': 424},
)

# Option – derive grid parameters from the bathymetry file
# dict_grid = case_grid.params_from_bathy()


print(f'The grid info dictionary for domain {DOMAIN_ID} is: {case_grid.dict_info}')
case_grid.fill_grid_section()


*** Initializing gridmaker for domain 1 ***

The grid info dictionary for domain 1 is: {'lon_ll_corner': 278.2265167, 'lat_ll_corner': 12.4413166, 'x_extent': 0.13185, 'y_extent': 0.1908, 'nx': 293, 'ny': 424}

 	*** Adding/Editing grid information for domain 1 in configuration file ***



### Setting up the bathymetry

`BathyMaker` class fills the `BOTTOM` section of `run.swn`. There is one method to set up the bathymetry file for the model:

* `use_ascii_file_from_user` is used when a pre-processed `.bot` file already exists in `input/domain_01/`. This method implies the use of a dictionary with bathymetry metadata.


```{note}
If the `use_ascii_file_from_user()` method is used, make sure that the pre-processed `.bot` file is correctly formatted and placed in the appropriate `input/domain_0X/` folder before running the script. This method will link or copy the existing `.bot` file to the run directory and update the bathymetry information in the configuration metadata, but it does not perform any processing on the file itself.

In [6]:
case_bathy = BathyMaker(
    init         = case,
    domain_number= DOMAIN_ID,
    dict_info   = {'lon_ll_corner_bot': 278.2265167, 'lat_ll_corner_bot': 12.4413166,
                    'nx_bot': 293, 'ny_bot': 424, 
                    'dx_bot': 0.00045, 'dy_bot': 0.00045},     
)

case_bathy.use_ascii_file_from_user()

print(f'The bathymetry info dictionary for domain {DOMAIN_ID} is: {case_bathy.dict_info}')
case_bathy.fill_bathy_section()


*** Initializing bathymaker for domain 1 ***

The bathymetry info dictionary for domain 1 is: {'lon_ll_corner_bot': 278.2265167, 'lat_ll_corner_bot': 12.4413166, 'nx_bot': 293, 'ny_bot': 424, 'dx_bot': 0.00045, 'dy_bot': 0.00045, 'bathy_file': '../../input/domain_01/bathymetry.bot'}

 	*** Adding/Editing bathymetry information for domain 1 in configuration file ***



```{hint}
If you already have a ready-to-use `.bot` file in `input/domain_01/`, call `case_bathy.use_ascii_file_from_user()` instead of `convert_xyz2asc`. Pass the metadata dict (or `None`) as `bathy_info=` at construction time and use `use_link=True` to symlink rather than copy the file into the run directory.

### Defining bottom friction information

`BottomFrictionProcessor` is a class inside `swanpy` that fills the `FRICTION` section of `run.swn` with the appropriate parameters for bottom friction.

In [7]:
case_bottom_friction = BottomFrictionProcessor(
    init         = case,
    domain_number= DOMAIN_ID,
    dict_info    = 
        {'lon_ll_corner_fric': 278.2265167, 'lat_ll_corner_fric': 12.4413166,
         'nx_fric': 293, 'ny_fric': 424,
         'dx_fric': 0.00045, 'dy_fric': 0.00045}
    )  # using grid info to fill friction section


case_bottom_friction.use_ascii_file_from_user()
print(case_bottom_friction.dict_info)
case_bottom_friction.fill_friction_section()


*** Initializing BottomFrictionProcessor for domain 1 ***

{'lon_ll_corner_fric': 278.2265167, 'lat_ll_corner_fric': 12.4413166, 'nx_fric': 293, 'ny_fric': 424, 'dx_fric': 0.00045, 'dy_fric': 0.00045, 'friction_file': '../../input/domain_01/kn.fric'}

 	*** Adding/Editing friction information for domain 1 in configuration file ***



### Defining wind forcing

`WindForcing` handles the retrieval of wind data, it converts it to the required format, and writes the `WIND` section of `run.swn`. For this scenario the data is not downloaded from any external source and it is explicitly provided as constant using the method `use_constant_wind()`. This method requieres the wind speed (in m/s) and direction (in degrees, where 0° is North, 90° is East, etc.) to be provided as arguments. 

In [8]:
case_winds = WindForcing(
    init         = case,
    domain_number= DOMAIN_ID,
)

case_winds.use_constant_wind(wind_speed=12.8, wind_dir=90)

print(f'The wind info dictionary for domain {DOMAIN_ID} is: {case_winds.dict_info}')

case_winds.fill_wind_section()


*** Initializing winds for domain 1 ***

The wind info dictionary for domain 1 is: {'wind_speed': 12.8, 'wind_dir': 90}

 	*** Adding/Editing winds information for domain 1 in configuration file ***



### Defining boundary conditions

`BoundaryConditions` writes the `BOUNDARY` section of `run.swn`. For this stationary case, the boudnary conditions are defined through the sides of the domains so that informations has to be provided as a dictionary called ``dict_info`` at construction time. This dictionary is expected to have the following keys:
- `boundary_type`: A string with the type of boundary condition ("side" or "segment")

In [9]:
case_boundary = BoundaryConditions(
    init        = case,
    domain_number= DOMAIN_ID,
    dict_info   = {"boundary_type": "side"})

KeyError: 'parent_domains'

Once the class is initialized, the method `create_boundary_line()` can be used to define the boundary conditions along the sides of the domain. This method requires a list of sides (e.g., ["N", "S", "E", "W"]) and a dictionary with wave parameters (e.g., {'hs': 1.5, 'tp': 8, 'dir': 90}) to be provided as arguments. The wave parameters include significant wave height (`hs`), peak period (`tp`), and wave direction (`dir`).

In [ ]:
case_boundary.create_boundary_line(list_sides=["N"],wave_params={'hs': 1.7, 'tp': 9, 'dir': 90 , 'spread':2})
case_boundary.fill_boundaries_section()

### Computation and run configuration

Finally, after all the input data is set up, the `CaseRunner` class handles the final steps to finish the configuration of the case and prepare it for running:

* `define_output_from_file()` – reads a CSV of output point coordinates and writes a `.loc` file.
* `fill_computation_section()` – writes the `COMPUTE` command (stationary or non-stationary).
* `fill_slurm_file()` – (HPC only) generates a SLURM launcher script.


`CaseRunner` expects a dictionary of computation parameters (`dict_comp_data`) that contains all the necessary information to write the `COMPUTE` section of `run.swn`. All the supported keys are listed below. 

- `stat_comp`: 1 for stationary, 0 for non-stationary
- `comp_dates`: list of datetime objects for computation dates
- `init_intermediate`: boolean to insert INIT commands between snapshots



For a stationary run, the following keys are expected in this dictionary:

In [ ]:
comp_data_stat = dict(
    stat_comp        = 1,               # 1 → stationary
)

Finally the CaseRunner is instantiated with the initialized case and the computation parameters. Then the `define_output_from_file()` and `fill_computation_section()` methods are called to write the output definition and the COMPUTE command in run.swn, respectively. The case is now ready to be run with SWAN.

In [ ]:
case_run = CaseRunner(
    init         = case,
    domain_number= DOMAIN_ID,
    dict_comp_data= comp_data_stat
)

# define point output locations from a CSV with X, Y columns
case_run.define_output_from_file(filename='output_points.csv')

# Write the COMPUTE command and clean up any remaining template placeholders
case_run.fill_computation_section()

# (Optional) generate SLURM launcher for HPC submission
# case_run.fill_slurm_file()

All the information ready to be used for the stationary case is now set up and it can be found in the `run/domain_01` folder- The next step is to run the case using the swan executable on the HPC or locally, depending on the setup. 

### Reading the output

After running the case, the output files will be generated in the `output/domain_01/` folder, and they can be post-processed and visualized as needed.